# 6. Proposition Chunking

Rewrite passages into atomic, self-contained facts (propositions) and index those.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

A normal chunk mixes several facts and has pronouns like *it* or *this model* whose meaning is only clear from earlier text. **Propositional chunking** (Dense X Retrieval) rewrites text into atomic, self-contained facts. Each proposition is one idea, fully de-referenced, so its embedding is sharp and it can match a specific question precisely.

## How it works

1. Take each chunk
2. Ask the LLM to decompose it into atomic statements and resolve pronouns
3. Embed each proposition separately
4. Search propositions; map hits back to their parent chunk for context
5. Generate from the parent chunks (or the propositions themselves)

```text
 chunk: "It produces 384-dimensional vectors and is popular because it is small."
   -> LLM ->  "all-MiniLM-L6-v2 produces 384-dimensional vectors."
              "all-MiniLM-L6-v2 is popular for local experiments because it is small and fast."
   -> embed each -> search -> parent chunk -> LLM
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

A **proposition** is a single fact that makes sense on its own - pronouns resolved, no hidden references.

In [ ]:
def propositions(chunk):
    data = llm_json(
        "Decompose the passage into short, self-contained factual statements. Replace pronouns with the "
        "entity they refer to. Reply with a JSON list of strings.\n\nPassage: " + chunk
    )
    return [p for p in (data or []) if isinstance(p, str)]

print(propositions(chunks[0]))

In [ ]:
props, src = [], []
for i, chunk in enumerate(chunks):
    for p in propositions(chunk):
        props.append(p)
        src.append({"chunk": i})
prop_store = build_store(props, src)
print(len(props), "propositions")

In [ ]:
query = "What is the constant in the Reciprocal Rank Fusion formula?"
print("--- propositions"); show(prop_store.search(query, 3))
print("--- plain chunks"); show(store.search(query, 3))

## Quality control
LLM decomposition can drop facts or add new ones. Spot-check by comparing a chunk with its propositions.

In [ ]:
i = 1
print("CHUNK:\n", chunks[i], "\n")
for p in propositions(chunks[i]): print(" -", p)

## Do propositions retrieve better than chunks?
Propositions are paraphrased, so substring matching on them would be unfair. We map each proposition hit back to its parent chunk and check the evidence there.

In [ ]:
def prop_search(query, k=3):
    seen, out = set(), []
    for h in prop_store.search(query, k * 4):
        c = h["meta"]["chunk"]
        if c not in seen:
            seen.add(c); out.append({"idx": c, "text": chunks[c], "score": h["score"], "meta": {}})
    return out[:k]

compare_retrievers({"fixed chunks (500)": store.search, "propositions -> parent chunk": prop_search}, k=3)

## Inspect the index
How many propositions per chunk, and how long are they?

In [ ]:
import pandas as pd
pd.DataFrame({"chars": [len(p) for p in props]}).describe().round(1).T

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Sharper embeddings; precise matches for specific facts | Expensive: one LLM call per chunk at indexing time |
| Pronouns and references are resolved | Risk of LLM dropping or inventing facts |
| Great for question answering over dense factual text | Index size grows 3-10x |

## When to use it

Fact-dense corpora (encyclopaedic, policy, technical docs) where users ask narrow factual questions and indexing cost is acceptable.

## Experiments to try

- Add a second LLM pass that grades each proposition for accuracy/completeness.
- Retrieve propositions but send the parent chunk to the generator.
- Compare index size and hit rate against fixed chunks.

## Key takeaways

- Atomic, de-referenced facts embed better.
- Always keep a pointer to the source chunk.
- Validate LLM-generated propositions.

---
**Next:** [7. Evaluation with DeepEval](../07_evaluation_deep_eval/evaluation_deep_eval.ipynb)